In [8]:
import pandas as pd
import numpy as np
import random
from faker import Faker
import os
from datetime import timedelta, datetime

fake = Faker()
Faker.seed(42)



In [ ]:
def generate_synthetic_sales_data(
    n_rows=1000, 
    seed=42, 
    pct_no_purchase=0.4,
    start_year=2015, # Start date for generating order dates
    end_year=2024,   # End date for generating order dates
    peak_month=12,   # Month with highest sales (e.g., 12 for December)
    seasonality_amplitude=0.3, # How strong seasonality is (e.g., +/- 30%)
    yearly_growth_rate=0.03 # Annual sales growth (e.g., 3% per year)
):
    np.random.seed(seed)
    random.seed(seed)

    ship_modes = ['First Class', 'Second Class', 'Standard Class', 'Same Day']
    segments = ['Consumer', 'Corporate', 'Home Office']
    categories = ['Furniture', 'Office Supplies', 'Technology']
    sub_categories = {
        'Furniture': ['Chairs', 'Tables', 'Bookcases'],
        'Office Supplies': ['Binders', 'Paper', 'Art'],
        'Technology': ['Phones', 'Accessories', 'Copiers']
    }
    products = {
        'Chairs': ['Ergonomic Chair', 'Mesh Chair'],
        'Tables': ['Conference Table', 'Office Desk'],
        'Bookcases': ['Wooden Bookcase', 'Metal Bookcase'],
        'Binders': ['3-Ring Binder', 'Heavy-Duty Binder'],
        'Paper': ['Printer Paper', 'Notebook Paper'],
        'Art': ['Markers', 'Sketch Pens'],
        'Phones': ['iPhone 14', 'Samsung Galaxy'],
        'Accessories': ['Mouse', 'Keyboard'],
        'Copiers': ['Canon Copier', 'HP Copier']
    }

    countries_by_region = {
        'North America': ['United States', 'Canada', 'Mexico'],
        'Europe': ['United Kingdom', 'Germany', 'France', 'Italy', 'Spain', 'Netherlands', 'Sweden'],
        'Asia': ['India', 'China', 'Japan', 'South Korea', 'Singapore', 'Indonesia'],
        'South America': ['Brazil', 'Argentina', 'Chile', 'Colombia'],
        'Africa': ['South Africa', 'Nigeria', 'Egypt', 'Kenya'],
        'Oceania': ['Australia', 'New Zealand']
    }
    all_countries = [country for region in countries_by_region.values() for country in region]

    avg_rows_per_customer = 100.8
    num_customers = int(np.ceil(n_rows / avg_rows_per_customer))

    # Generate unique customers with consistent names and IDs, using original ID pattern
    customers = []
    used_customer_ids = set()
    while len(customers) < num_customers:
        cust_id = f"{random.choice(['CG', 'SC', 'NG'])}-{random.randint(10000, 99999)}"
        if cust_id in used_customer_ids:
            continue
        used_customer_ids.add(cust_id)
        cust_name = fake.name()
        segment = random.choice(segments)
        country = random.choice(all_countries)
        region = [r for r, countries in countries_by_region.items() if country in countries][0]
        state = fake.state()
        city = fake.city()
        postal_code = fake.postcode()

        customers.append({
            'Customer ID': cust_id,
            'Customer Name': cust_name,
            'Segment': segment,
            'Country': country,
            'Region': region,
            'State': state,
            'City': city,
            'Postal Code': postal_code
        })
    customers_df = pd.DataFrame(customers)

    # Prepare customer indices repeated ~12.6 times per customer, ensuring length >= n_rows
    customer_indices = np.repeat(range(num_customers), int(avg_rows_per_customer))
    while len(customer_indices) < n_rows:
        customer_indices = np.concatenate([customer_indices, np.repeat(range(num_customers), int(avg_rows_per_customer))])
    customer_indices = customer_indices[:n_rows]
    np.random.shuffle(customer_indices)

    # Note: `n_no_purchase` value should typically be `int(n_rows * pct_no_purchase)`
    # The fixed value `5084` might not scale with `n_rows`. Let's use `pct_no_purchase`.
    n_no_purchase = int(n_rows * pct_no_purchase)
    no_purchase_indices = set(random.sample(range(n_rows), n_no_purchase))

    data = []

    for i in range(n_rows):
        cust = customers_df.iloc[customer_indices[i]]

        # Generate order date within the specified year range
        start_date_dt = datetime(start_year, 1, 1)
        end_date_dt = datetime(end_year, 12, 31)
        time_delta = end_date_dt - start_date_dt
        order_date = start_date_dt + timedelta(days=random.randint(0, time_delta.days))

        ship_date = order_date + timedelta(days=np.random.randint(1, 7))
        ship_mode = random.choice(ship_modes)

        category = random.choice(categories)
        sub_category = random.choice(sub_categories[category])
        product_name = random.choice(products[sub_category])

        # --- Seasonality and Trend Calculation ---
        month_num = order_date.month
        year_num = order_date.year
        
        # Calculate phase shift to align peak_month with sine wave peak
        # Sine wave peaks at pi/2. So, we need (2 * pi * (peak_month - 1) / 12) to be pi/2
        # Which means (peak_month - 1) / 12 = 0.25 => peak_month - 1 = 3 => peak_month = 4 (April)
        # So, to shift peak to `peak_month`, we adjust the phase of the sine wave.
        # A full cycle is 12 months. peak_month relative to start of year (month 1).
        # We want `sin(angle)` to be 1 when month is `peak_month`.
        # `angle = 2 * pi * (month_num - 1 + (12 - peak_month + 0.25*12)) / 12`
        # Simpler: `np.sin(2 * np.pi * (month_num - (peak_month - 1)) / 12)`
        # This makes peak_month the 'start' of the sine wave cycle for calculation
        # To get peak at `peak_month`, adjust phase: `(month_num - peak_month_adj) / 12 * 2pi`
        # A simpler way to control peak: `np.cos(2 * np.pi * (month_num - peak_month) / 12)`
        # Cosine peaks at 0, so if month_num == peak_month, it's 0, cos(0)=1 (peak)
        
        seasonality_factor = 1.0 + seasonality_amplitude * np.cos(2 * np.pi * (month_num - peak_month) / 12)
        
        # Linear yearly growth
        years_since_start = year_num - start_year
        growth_factor = 1.0 + (years_since_start * yearly_growth_rate)

        if i in no_purchase_indices:
            quantity = 0
            sales = 0.0
            profit = 0.0
            discount = 0.0
        else:
            quantity = np.random.randint(1, 10)
            base_price = np.random.uniform(20, 500)
            discount = round(np.random.choice([0.0, 0.1, 0.2, 0.3]), 2)
            
            # Base sales before seasonality and growth
            raw_sales = quantity * base_price * (1 - discount)
            
            # Apply seasonality and growth
            sales = round(raw_sales * seasonality_factor * growth_factor, 2)
            
            # Ensure quantity also reflects seasonality and growth somewhat
            # Scale quantity using a slightly dampened seasonal/growth factor and noise
            quantity = max(1, round(quantity * seasonality_factor * growth_factor * np.random.uniform(0.9, 1.1)))

            # Profit still derived from final sales, but with its own random factor
            profit = round(sales * np.random.uniform(0.05, 0.3), 2)

        country_code = ''.join([c[0] for c in cust['Country'].split()])[:2].upper()
        order_id = f"{country_code}-{order_date.year}-{random.randint(100000, 999999)}"
        # The Product ID generation is quite generic; consider making it more structured if Product ID
        # needs to be consistent for the same Category/Sub-Category/Product Name over time.
        # For now, it's okay as your model uses Product Name (string) which will be encoded.
        product_id = f"{category[:3].upper()}-{sub_category[:2].upper()}-{random.randint(10000000, 99999999)}"

        data.append([
            i + 1,
            order_id,
            order_date.strftime('%Y-%m-%d'),
            ship_date.strftime('%Y-%m-%d'),
            ship_mode,
            cust['Customer ID'],
            cust['Customer Name'],
            cust['Segment'],
            cust['Country'],
            cust['City'],
            cust['State'],
            cust['Postal Code'],
            cust['Region'],
            product_id,
            category,
            sub_category,
            product_name,
            round(sales, 2), # Ensure sales is rounded after all factors
            quantity,        # Ensure quantity is rounded
            discount,
            round(profit, 2)
        ])

    columns = ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
               'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
               'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
               'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

    print(f"Generated {n_rows} rows of synthetic data.")
    return pd.DataFrame(data, columns=columns)


In [3]:

def generate_synthetic_sales_data(n_rows=1000, seed=42, pct_no_purchase=0.4):
    np.random.seed(seed)
    random.seed(seed)

    ship_modes = ['First Class', 'Second Class', 'Standard Class', 'Same Day']
    segments = ['Consumer', 'Corporate', 'Home Office']
    categories = ['Furniture', 'Office Supplies', 'Technology']
    sub_categories = {
        'Furniture': ['Chairs', 'Tables', 'Bookcases'],
        'Office Supplies': ['Binders', 'Paper', 'Art'],
        'Technology': ['Phones', 'Accessories', 'Copiers']
    }
    products = {
        'Chairs': ['Ergonomic Chair', 'Mesh Chair'],
        'Tables': ['Conference Table', 'Office Desk'],
        'Bookcases': ['Wooden Bookcase', 'Metal Bookcase'],
        'Binders': ['3-Ring Binder', 'Heavy-Duty Binder'],
        'Paper': ['Printer Paper', 'Notebook Paper'],
        'Art': ['Markers', 'Sketch Pens'],
        'Phones': ['iPhone 14', 'Samsung Galaxy'],
        'Accessories': ['Mouse', 'Keyboard'],
        'Copiers': ['Canon Copier', 'HP Copier']
    }

    countries_by_region = {
        'North America': ['United States', 'Canada', 'Mexico'],
        'Europe': ['United Kingdom', 'Germany', 'France', 'Italy', 'Spain', 'Netherlands', 'Sweden'],
        'Asia': ['India', 'China', 'Japan', 'South Korea', 'Singapore', 'Indonesia'],
        'South America': ['Brazil', 'Argentina', 'Chile', 'Colombia'],
        'Africa': ['South Africa', 'Nigeria', 'Egypt', 'Kenya'],
        'Oceania': ['Australia', 'New Zealand']
    }
    all_countries = [country for region in countries_by_region.values() for country in region]

    avg_rows_per_customer = 100.8
    num_customers = int(np.ceil(n_rows / avg_rows_per_customer))

    # Generate unique customers with consistent names and IDs, using original ID pattern
    customers = []
    used_customer_ids = set()
    while len(customers) < num_customers:
        cust_id = f"{random.choice(['CG', 'SC', 'NG'])}-{random.randint(10000, 99999)}"
        if cust_id in used_customer_ids:
            continue
        used_customer_ids.add(cust_id)
        cust_name = fake.name()
        segment = random.choice(segments)
        country = random.choice(all_countries)
        region = [r for r, countries in countries_by_region.items() if country in countries][0]
        state = fake.state()
        city = fake.city()
        postal_code = fake.postcode()

        customers.append({
            'Customer ID': cust_id,
            'Customer Name': cust_name,
            'Segment': segment,
            'Country': country,
            'Region': region,
            'State': state,
            'City': city,
            'Postal Code': postal_code
        })
    customers_df = pd.DataFrame(customers)

    # Prepare customer indices repeated ~12.6 times per customer, ensuring length >= n_rows
    customer_indices = np.repeat(range(num_customers), int(avg_rows_per_customer))
    while len(customer_indices) < n_rows:
        customer_indices = np.concatenate([customer_indices, np.repeat(range(num_customers), int(avg_rows_per_customer))])
    customer_indices = customer_indices[:n_rows]
    np.random.shuffle(customer_indices)

    n_no_purchase = 5084 # int(n_rows * pct_no_purchase)
    no_purchase_indices = set(random.sample(range(n_rows), n_no_purchase))

    data = []

    for i in range(n_rows):
        cust = customers_df.iloc[customer_indices[i]]

        order_date = fake.date_between(start_date='-10y', end_date='today')
        ship_date = order_date + timedelta(days=np.random.randint(1, 7))
        ship_mode = random.choice(ship_modes)

        category = random.choice(categories)
        sub_category = random.choice(sub_categories[category])
        product_name = random.choice(products[sub_category])

        if i in no_purchase_indices:
            quantity = 0
            sales = 0.0
            profit = 0.0
            discount = 0.0
        else:
            quantity = np.random.randint(1, 10)
            base_price = np.random.uniform(20, 500)
            discount = round(np.random.choice([0.0, 0.1, 0.2, 0.3]), 2)
            sales = round(quantity * base_price * (1 - discount), 2)
            profit = round(sales * np.random.uniform(0.05, 0.3), 2)

        country_code = ''.join([c[0] for c in cust['Country'].split()])[:2].upper()
        order_id = f"{country_code}-{order_date.year}-{random.randint(100000, 999999)}"
        product_id = f"{category[:3].upper()}-{sub_category[:2].upper()}-{random.randint(10000000, 99999999)}"

        data.append([
            i + 1,
            order_id,
            order_date.strftime('%Y-%m-%d'),
            ship_date.strftime('%Y-%m-%d'),
            ship_mode,
            cust['Customer ID'],
            cust['Customer Name'],
            cust['Segment'],
            cust['Country'],
            cust['City'],
            cust['State'],
            cust['Postal Code'],
            cust['Region'],
            product_id,
            category,
            sub_category,
            product_name,
            round(sales, 2),
            quantity,
            discount,
            round(profit, 2)
        ])

    columns = ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
               'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
               'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
               'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

    return pd.DataFrame(data, columns=columns)


In [5]:
# data_df = generate_synthetic_sales_data(n_rows=4997*5)
# data_df.to_csv('../data/raw_data/Sample - Superstore (Synthetic).csv',index=False)

raw_df = pd.read_csv(f'{os.getcwd()}/../data/raw_data/Sample - Superstore.csv', encoding='windows-1254')
# synthetic_raw_df = generate_synthetic_sales_data(n_rows=4997*5)
# complete_raw_df = pd.concat([raw_df, synthetic_raw_df], ignore_index=True)
# complete_raw_df = complete_raw_df.sample(frac=1, random_state=42).reset_index(drop=True)
# complete_raw_df.to_csv('../data/raw_data/Sample - Superstore (Complete).csv',encoding='utf-8')

raw_df['Order Date'] = pd.to_datetime(raw_df['Order Date'], format='mixed')
min(raw_df['Order Date'].dt.year)


2014

In [8]:
data_df['purchase_flag'] = data_df['Quantity']>0
data_df['purchase_flag'].value_counts()


purchase_flag
True     19901
False     5084
Name: count, dtype: int64

In [12]:
# import random

# rand_num = random.randint(7500, 10000)
# print(rand_num)
4997*8


synthetic_df_seasonality = generate_synthetic_sales_data(
        n_rows=4997*8, # Increased rows for better seasonality visibility
        start_year=2014,
        end_year=2024,
        peak_month=12, # Peak sales in December
        seasonality_amplitude=0.2, # Stronger seasonality
        yearly_growth_rate=0.05 # 5% annual growth
    )

raw_df = pd.read_csv(f'{os.getcwd()}/../data/raw_data/Sample - Superstore.csv', encoding='windows-1254')
# synthetic_raw_df = generate_synthetic_sales_data(n_rows=4997*5)
complete_raw_df = pd.concat([raw_df, synthetic_df_seasonality], ignore_index=True)
complete_raw_df = complete_raw_df.sample(frac=1, random_state=42).reset_index(drop=True)
complete_raw_df.to_csv('../data/raw_data/Complete.csv',encoding='utf-8')



Generated 39976 rows of synthetic data.


In [40]:

# Initialize Faker for realistic-looking data
# Initialize Faker with multiple locales if needed, or specify a default.
# For specific locale attributes like 'province', we will access them directly.
fake = Faker(['en_US', 'en_IN', 'en_GB', 'en_CA', 'en_AU', 'de_DE'])

# --- State Abbreviation to Full Name Mappings ---
# For India
IN_STATES_MAP = {
    'AP': 'Andhra Pradesh',
    'MH': 'Maharashtra',
    'KA': 'Karnataka',
    'DL': 'Delhi',
    'TN': 'Tamil Nadu',
    'UP': 'Uttar Pradesh',
    'WB': 'West Bengal',
    'GJ': 'Gujarat',
    'RJ': 'Rajasthan',
    'MP': 'Madhya Pradesh'
}

# For USA
US_STATES_MAP = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

# --- Global Configuration Parameters ---
# Date range for transactions and product release dates
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2025, 5, 30)

# Electronics Categories and Subcategories with typical price ranges (base price)
ELECTRONICS_SUBCATEGORIES = {
    'Smartphones': {'brands': ['Apple', 'Samsung', 'Google', 'Xiaomi'], 'price_range': (400.00, 1500.00)},
    'Laptops': {'brands': ['Apple', 'Dell', 'HP', 'Lenovo', 'Acer'], 'price_range': (600.00, 2500.00)},
    'Televisions': {'brands': ['Samsung', 'LG', 'Sony', 'TCL'], 'price_range': (300.00, 3000.00)},
    'Wearables': {'brands': ['Apple', 'Samsung', 'Garmin', 'Fitbit'], 'price_range': (100.00, 500.00)},
    'Audio Devices': {'brands': ['Sony', 'JBL', 'Bose', 'Sennheiser'], 'price_range': (50.00, 800.00)},
    'Gaming Consoles': {'brands': ['Sony', 'Microsoft', 'Nintendo'], 'price_range': (300.00, 600.00)},
    'Cameras': {'brands': ['Canon', 'Nikon', 'Sony', 'Fujifilm'], 'price_range': (200.00, 1200.00)},
    'Tablets': {'brands': ['Apple', 'Samsung', 'Microsoft'], 'price_range': (200.00, 1000.00)},
    'Computer Components': {'brands': ['Intel', 'AMD', 'Nvidia', 'Corsair'], 'price_range': (50.00, 1000.00)},
    'Peripherals': {'brands': ['Logitech', 'Razer', 'Corsair', 'Microsoft'], 'price_range': (20.00, 200.00)},
}

# Customer demographics
AGE_GROUPS = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
GENDERS = ['Male', 'Female', 'Other']
INCOME_LEVELS = ['Low', 'Medium', 'High']
COUNTRIES = ['USA', 'India', 'UK', 'Canada', 'Australia', 'Germany']
US_STATES_ABBR = list(US_STATES_MAP.keys())
IN_STATES_ABBR = list(IN_STATES_MAP.keys())
UK_CITIES = ['London', 'Manchester', 'Birmingham']
CA_CITIES = ['Toronto', 'Vancouver', 'Montreal']
AU_CITIES = ['Sydney', 'Melbourne', 'Brisbane']
DE_CITIES = ['Berlin', 'Munich', 'Hamburg']
CUSTOMER_SEGMENTS = ['Tech Enthusiast', 'Budget Buyer', 'Premium Shopper', 'Casual Buyer', 'New Customer']

# Payment methods and device types
PAYMENT_METHODS = ['Credit Card', 'Debit Card', 'UPI', 'Net Banking', 'PayPal']
DEVICE_TYPES = ['Mobile App', 'Website (Desktop)', 'Website (Tablet)', 'In-store POS']

# Holiday seasons (simplified for demonstration)
HOLIDAYS = [
    (1, 1, 'New Year\'s Day', True),
    (11, 28, 'Thanksgiving', True), # Approx. 4th Thursday in Nov
    (11, 29, 'Black Friday', True),
    (12, 25, 'Christmas', True),
    (10, 24, 'Diwali', True), # Approx. Oct/Nov, varies by year
    (7, 4, 'Independence Day (US)', True),
    (1, 26, 'Republic Day (India)', False),
    (8, 15, 'Independence Day (India)', False),
    (12, 26, 'Boxing Day', True),
]

# Product lifecycle stages
LIFECYCLE_STAGES = ['New Launch', 'Growth', 'Mature', 'Decline', 'End-of-Life']



In [46]:

def generate_customer_data(num_customers):
    """
    Generates a DataFrame of synthetic customer data.
    Uses global configuration parameters.
    """
    customers = []
    for i in range(num_customers):
        country = random.choice(COUNTRIES)
        state_province_full = None
        zip_code = None

        if country == 'USA':
            state_abbr = random.choice(US_STATES_ABBR)
            state_province_full = US_STATES_MAP[state_abbr]
            city = fake.city()
            # Explicitly use en_US locale for zipcode_in_state
            zip_code = fake['en_US'].zipcode_in_state(state_abbr)
        elif country == 'India':
            state_abbr = random.choice(IN_STATES_ABBR)
            state_province_full = IN_STATES_MAP[state_abbr]
            city = fake.city()
            zip_code = fake.postcode()
        elif country == 'UK':
            state_province_full = 'England'
            city = random.choice(UK_CITIES)
            zip_code = fake.postcode()
        elif country == 'Canada':
            # Access Canadian locale for province
            state_province_full = fake['en_CA'].province()
            city = random.choice(CA_CITIES)
            zip_code = fake['en_CA'].postcode() # Use Canadian postcode for Canada
        elif country == 'Australia':
            # Access Australian locale for state
            state_province_full = fake['en_AU'].state()
            city = random.choice(AU_CITIES)
            zip_code = fake['en_AU'].postcode() # Use Australian postcode for Australia
        elif country == 'Germany':
            # Access German locale for state
            state_province_full = fake['de_DE'].state()
            city = random.choice(DE_CITIES)
            zip_code = fake['de_DE'].postcode() # Use German postcode for Germany
        else: # Fallback
            state_province_full = fake.state() # Default state if locale not specified
            city = fake.city()
            zip_code = fake.postcode()

        customers.append({
            'customer_id': f'CUST{i:05d}',
            'customer_age_group': random.choice(AGE_GROUPS),
            'customer_gender': random.choice(GENDERS),
            'customer_income_level': random.choice(INCOME_LEVELS),
            'customer_country': country,
            'customer_state_province': state_province_full,
            'customer_city': city,
            'customer_zip_postal_code': zip_code,
            'customer_segment': random.choice(CUSTOMER_SEGMENTS),
            'customer_lifetime_value': round(random.uniform(0, 5000), 2),
            'customer_purchase_frequency': random.randint(0, 20),
            'time_since_last_purchase_days': random.randint(0, 365) if i > 0 else None,
        })
    return pd.DataFrame(customers)


In [51]:

def generate_product_data(num_products):
    """
    Generates a DataFrame of synthetic electronics product data.
    Uses global configuration parameters.
    """
    products = []
    product_counter = 0
    for subcategory, details in ELECTRONICS_SUBCATEGORIES.items():
        for brand in details['brands']:
            for _ in range(random.randint(1, 3)):
                product_id = f'ELEC{product_counter:04d}'
                product_name = f'{brand} {subcategory} {fake.word().capitalize()}'
                base_price = round(random.uniform(details['price_range'][0], details['price_range'][1]), 2)
                current_inventory = random.randint(10, 500)
                avg_lead_time = random.randint(2, 20)
                release_date = fake.date_between(start_date=datetime(2020, 1, 1), end_date=datetime(2024, 12, 31))
                lifecycle_stage = random.choice(LIFECYCLE_STAGES)

                products.append({
                    'product_id': product_id,
                    'product_name': product_name,
                    'product_category': 'Electronics',
                    'product_subcategory': subcategory,
                    'product_brand': brand,
                    'product_base_price': base_price,
                    'current_inventory_level': current_inventory,
                    'average_supplier_lead_time_days': avg_lead_time,
                    'product_release_date': release_date,
                    'product_lifecycle_stage': lifecycle_stage,
                })
                product_counter += 1
    return pd.DataFrame(products)


In [52]:

def generate_transaction_data(num_records, customers_df, products_df):
    """
    Generates a DataFrame of raw synthetic transaction data, including derived temporal and behavioral features.
    It takes pre-generated customer and product DataFrames to link transactions.
    Uses global configuration parameters.
    """
    transactions_data = []
    for i in range(num_records):
        transaction_id = f'TXN{i:06d}'
        
        customer = customers_df.sample(1).iloc[0]
        product = products_df.sample(1).iloc[0]

        time_delta = END_DATE - START_DATE
        random_seconds = random.randint(0, int(time_delta.total_seconds()))
        transaction_datetime = START_DATE + timedelta(seconds=random_seconds)

        quantity_purchased = 1
        if product['product_subcategory'] in ['Audio Devices', 'Peripherals'] and random.random() < 0.2:
            quantity_purchased = random.randint(1, 2)

        item_price_at_purchase = round(product['product_base_price'] * random.uniform(0.85, 1.0), 2)
        item_total_value = round(item_price_at_purchase * quantity_purchased, 2)
        
        total_transaction_value = item_total_value + round(random.uniform(0, 10), 2)

        transaction_year = transaction_datetime.year
        transaction_month = transaction_datetime.month
        transaction_day_of_week = transaction_datetime.isoweekday()
        transaction_day_of_month = transaction_datetime.day
        transaction_week_of_year = transaction_datetime.isocalendar()[1]

        is_holiday_season = False
        holiday_type = None
        days_until_next_major_holiday = None

        for h_month, h_day, h_name, h_major in HOLIDAYS:
            if h_name == 'Diwali':
                if transaction_month == 10 and 20 <= transaction_day_of_month <= 28:
                     is_holiday_season = True
                     holiday_type = h_name
                     next_holiday_date = datetime(transaction_year, 10, 24)
                     if next_holiday_date < transaction_datetime:
                         next_holiday_date = datetime(transaction_year + 1, 10, 24)
                     days_until_next_major_holiday = (next_holiday_date - transaction_datetime).days
                     break
            elif transaction_month == h_month and transaction_day_of_month == h_day:
                if h_major:
                    is_holiday_season = True
                    holiday_type = h_name
                next_holiday_date = datetime(transaction_year, h_month, h_day)
                if next_holiday_date < transaction_datetime:
                    next_holiday_date = datetime(transaction_year + 1, h_month, h_day)
                days_until_next_major_holiday = (next_holiday_date - transaction_datetime).days
                break

        if transaction_month in [12, 1, 2]: season_of_year = 'Winter'
        elif transaction_month in [3, 4, 5]: season_of_year = 'Spring'
        elif transaction_month in [6, 7, 8]: season_of_year = 'Summer'
        else: season_of_year = 'Autumn'

        product_views_in_session = random.randint(1, 10)
        add_to_cart_events_in_session = 1 if random.random() < 0.7 else 0
        search_queries_in_session = random.randint(0, 5)
        wishlist_add_events_overall = random.randint(0, 3)

        transactions_data.append({
            'transaction_id': transaction_id,
            'customer_id': customer['customer_id'],
            'product_id': product['product_id'],
            'transaction_datetime': transaction_datetime,
            'quantity_purchased': quantity_purchased,
            'item_price_at_purchase': item_price_at_purchase,
            'item_total_value': item_total_value,
            'total_transaction_value': total_transaction_value,
            'payment_method': random.choice(PAYMENT_METHODS),
            'device_type': random.choice(DEVICE_TYPES),
            'transaction_year': transaction_year,
            'transaction_month': transaction_month,
            'transaction_day_of_week': transaction_day_of_week,
            'transaction_day_of_month': transaction_day_of_month,
            'transaction_week_of_year': transaction_week_of_year,
            'is_holiday_season': is_holiday_season,
            'holiday_type': holiday_type,
            'days_until_next_major_holiday': days_until_next_major_holiday,
            'season_of_year': season_of_year,
            'product_views_in_session': product_views_in_session,
            'add_to_cart_events_in_session': add_to_cart_events_in_session,
            'search_queries_in_session': search_queries_in_session,
            'wishlist_add_events_overall': wishlist_add_events_overall,
            'customer_lifetime_value_at_txn': customer['customer_lifetime_value'],
            'customer_purchase_frequency_at_txn': customer['customer_purchase_frequency'],
            'time_since_last_purchase_days_at_txn': customer['time_since_last_purchase_days']
        })
    
    return pd.DataFrame(transactions_data)


In [55]:

def generate_full_electronics_dataset(num_records=1000):
    """
    Orchestrates the generation of customer, product, and transaction data,
    and merges them into a single, denormalized DataFrame.
    Uses global configuration parameters.
    """

    # --- 1. Generate Customer Data ---
    num_customers = int(num_records * 0.3)
    customers_df = generate_customer_data(num_customers)

    # --- 2. Generate Product Data ---
    num_products = int(num_records * 0.1)
    products_df = generate_product_data(num_products)

    # --- 3. Generate Raw Transaction Data ---
    transactions_df = generate_transaction_data(num_records, customers_df, products_df)

    # --- 4. Merge all three datasets ---
    # Merge transactions with customer details.
    # No suffixes needed here as customer_lifetime_value_at_txn is distinct from customer_lifetime_value
    merged_df = pd.merge(transactions_df, customers_df, on='customer_id', how='left')

    # Merge the result with product details.
    final_df = pd.merge(merged_df, products_df, on='product_id', how='left')

    # Clean up and finalize columns
    # Use the 'customer_lifetime_value' and 'customer_purchase_frequency' columns
    # that came directly from customers_df during the merge.
    final_df['customer_lifetime_value'] = final_df['customer_lifetime_value'] + final_df['item_total_value']
    final_df['customer_purchase_frequency'] = final_df['customer_purchase_frequency'] + 1
    final_df['time_since_last_purchase_days'] = final_df['time_since_last_purchase_days'].fillna(365)

    # Drop the temporary '_at_txn' columns as they represent values *before* this transaction
    # and we have now calculated the *updated* values or used the current customer values.
    columns_to_drop = [
        'customer_lifetime_value_at_txn',
        'customer_purchase_frequency_at_txn',
        'time_since_last_purchase_days_at_txn'
    ]
    final_df = final_df.drop(columns=columns_to_drop, errors='ignore')

    return final_df


In [ ]:

# Generate 5000 records
synthetic_df = generate_full_electronics_dataset(num_records=10000)

# # Display the first few rows
# print("Generated Synthetic Data (First 5 Rows):")
# print(synthetic_df.head())

# Display some basic statistics to verify
print("\nData Info:")
synthetic_df.info()

# print("\nValue Counts for Product Subcategory:")
# print(synthetic_df['product_subcategory'].value_counts())

# print("\nValue Counts for Customer Segment:")
# print(synthetic_df['customer_segment'].value_counts())

# print("\nAverage Item Price by Product Subcategory:")
# print(synthetic_df.groupby('product_subcategory')['item_price_at_purchase'].mean().round(2))

# Save to CSV
synthetic_df.to_csv('../data/synthetic_electronics_data.csv', index=False)
print("\nSynthetic data saved to 'synthetic_electronics_data.csv'")



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 43 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   transaction_id                   10000 non-null  object        
 1   customer_id                      10000 non-null  object        
 2   product_id                       10000 non-null  object        
 3   transaction_datetime             10000 non-null  datetime64[ns]
 4   quantity_purchased               10000 non-null  int64         
 5   item_price_at_purchase           10000 non-null  float64       
 6   item_total_value                 10000 non-null  float64       
 7   total_transaction_value          10000 non-null  float64       
 8   payment_method                   10000 non-null  object        
 9   device_type                      10000 non-null  object        
 10  transaction_year                 10000 non-null

In [ ]:
os.getcwd()


NameError: name 'os' is not defined